In [2]:
from database import MBSE 
import pandas as pd
from sqlalchemy import inspect
import mbse

def sqlalchemy_schema_to_df(base, include_constraints=True,):
    """Convert SQLAlchemy declarative metadata into a DataFrame."""
    metadata = base.metadata
    rows = []
    for table in metadata.sorted_tables:
        # Identify foreign keys by column.
        foreign_keys = {fk.parent.name: (f"{fk.target_fullname}") for fk in table.foreign_keys}
        # Identify columns belonging to unique constraints.
        unique_columns = set()
        for constraint in table.constraints:
            if constraint.__class__.__name__ == "UniqueConstraint":
                unique_columns.update(column.name for column in constraint.columns)
        # Identify indexed columns.
        indexed_columns = set()
        for index in table.indexes:
            indexed_columns.update(column.name for column in index.columns)
        for column in table.columns:
            rows.append({
                "schema": table.schema,
                "table": table.name,
                "column": column.name,
                "type": str(column.type),
                "nullable": column.nullable,
                "primary_key": column.primary_key,
                "foreign_key": foreign_keys.get(column.name),
                "unique": column.name in unique_columns,
                "indexed": column.name in indexed_columns,
                "default": (str(column.default.arg) if column.default is not None else None),
                "server_default": str(column.server_default.arg) if column.server_default is not None else None
            })
    return pd.DataFrame(rows)

df_schema = sqlalchemy_schema_to_df(MBSE)
print(df_schema.to_string(index=False))

Empty DataFrame
Columns: []
Index: []


In [3]:
from database import MBSE
import mbse

print(MBSE)
print(MBSE.metadata)
print(MBSE.metadata.sorted_tables)
print(list(MBSE.metadata.tables.keys()))

<class 'database.MBSE'>
MetaData()
[]
[]


In [5]:
from pathlib import Path

def print_directory_tree(
    directory: str | Path,
    prefix: str = "",
    ignore: set[str] | None = None
) -> None:
    """Recursively prints a visual directory tree starting from a given folder."""
    if ignore is None:
        # Common folders you usually don't want cluttering the output
        ignore = {".git", "__pycache__", ".venv", "venv", "node_modules", ".DS_Store"}

    root = Path(directory)
    if not root.exists():
        print(f"Error: Path '{root}' does not exist.")
        return

    # Print the root folder name on the very first call
    if prefix == "":
        print(f"{root.resolve().name}/")

    # Filter out ignored items and sort: directories first, then files alphabetically
    items = [p for p in root.iterdir() if p.name not in ignore]
    items.sort(key=lambda p: (not p.is_dir(), p.name.lower()))

    for index, item in enumerate(items):
        is_last = index == len(items) - 1
        connector = "└── " if is_last else "├── "
        display_name = f"{item.name}/" if item.is_dir() else item.name

        print(f"{prefix}{connector}{display_name}")

        if item.is_dir():
            extension = "    " if is_last else "│   "
            print_directory_tree(item, prefix=prefix + extension, ignore=ignore)

# Example usage:
print_directory_tree("C:/Users/Drew/Documents/MyFiles/repo/database_work/prod/backend/app")

app/
├── api/
│   ├── govern.py
│   ├── ingest.py
│   └── schema.py
├── models/
│   ├── mbse/
│   │   ├── applied_stereotype.py
│   │   ├── class_taxonomy.py
│   │   ├── engineering_model.py
│   │   ├── instance_definition.py
│   │   ├── model_version.py
│   │   ├── object_definition.py
│   │   ├── object_instance_property.py
│   │   ├── object_relationship.py
│   │   ├── relationship_definition.py
│   │   ├── relationship_endpoint.py
│   │   ├── relationship_instance_property.py
│   │   └── stereotype.py
│   ├── __init__.py
│   ├── database.py
│   ├── govern.py
│   ├── ingest.py
│   └── schema.ipynb
├── services/
│   ├── govern.py
│   ├── ingest.py
│   └── validate.py
├── static/
└── templates/
    ├── base.html
    ├── dashboard.html
    └── element.html


In [4]:
from app.models.database import MBSE
import app.models

print("Tables:")
print(list(MBSE.metadata.tables.keys()))

ModuleNotFoundError: No module named 'app'

In [6]:
from database import MBSE
import mbse

print("Metadata:", id(MBSE.metadata))
print("Tables:", list(MBSE.metadata.tables.keys()))

import mbse.engineering_model as em

print("Model metadata:", id(em.EngineeringModel.metadata))
print("Same metadata:", em.EngineeringModel.metadata is MBSE.metadata)

Metadata: 2126941434128
Tables: []


ModuleNotFoundError: No module named 'app'